# Notebook 04 — Exposure Windows: When During Pregnancy Matters

**GeoAI for Climate, Environment and Health — Zimbabwe DHS 2015 Workshop**  
25 June 2026 · YSN Bulawayo

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why the *timing* of an environmental exposure during pregnancy determines its biological effect.
2. Derive trimester exposure windows from DHS birth dates and link them to Zimbabwe's seasonal climate cycles.
3. Quantify the overlap between pregnancy trimesters and the rainy/dry season as a function of conception month.
4. Compare first, second, and third trimester exposures between adverse and normal birth outcomes.

---

**Context.** Environmental epidemiology increasingly shows that the effect of an exposure depends on *when* it occurs during fetal development. Heat in the third trimester (T3) has different consequences than heat in T1 (organogenesis). In Zimbabwe, a child conceived in January — T1 falls in the hot-rainy season — has a very different environmental exposure profile from one conceived in August — T1 falls in the cool-dry season. The DHS records birth month, which, combined with a 9-month gestational period, lets us reconstruct approximate trimester-level exposure windows. Each trimester corresponds to approximately three calendar months.

---
## Section 1 — Load Data

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

DATA_DIR = pathlib.Path().resolve().parent / 'data'

births = pd.read_csv(DATA_DIR / 'zw_births.csv')

print(f"Births: {births.shape[0]:,} rows × {births.shape[1]} columns")
print()
print("Trimester exposure columns available:")
t_cols = [c for c in births.columns if c.endswith(('_t1', '_t2', '_t3'))]
print(t_cols)
print()
print("Exposure derivation:")
print("  birth_month - 9  = conception_month (stored in conception_month)")
print("  T1 = months 1–3 after conception  (lst_t1, rain_t1, ndvi_t1)")
print("  T2 = months 4–6 after conception  (lst_t2, rain_t2, ndvi_t2)")
print("  T3 = months 7–9 after conception  (lst_t3, rain_t3, ndvi_t3)")

In [ ]:
print("Low birth weight prevalence (low_birth_weight == 1):")
lbw_prev = births['low_birth_weight'].mean()
print(f"  {lbw_prev:.3f}  ({lbw_prev*100:.1f}%)")
print()
print("Births with recorded birth weight (non-null):")
n_bw = births['birth_weight_g'].notna().sum()
print(f"  {n_bw:,} of {len(births):,} ({n_bw/len(births)*100:.1f}%)")
print()
print("Trimester T3 LST summary:")
print(births['lst_t3'].describe().round(2))

---
## Section 2 — Worked Example: Zimbabwe Seasonality and Pregnancy

For each conception month, we compute the mean third-trimester (T3) temperature across all births with that conception month. T3 = months 7–9 of gestation; a baby conceived in month `c` reaches T3 during calendar months `c+6` to `c+8` (wrapping around year-end).

Zimbabwe's rainy season runs approximately November–March. A baby conceived November–December experiences T3 in August–October — the transition from cool-dry to hot-dry — representing high cumulative heat stress.

In [ ]:
RAINY_MONTHS = [11, 12, 1, 2, 3]  # Zimbabwe wet season

t3_by_conception = (
    births
    .groupby('conception_month')['lst_t3']
    .mean()
    .reset_index()
    .rename(columns={'lst_t3': 'mean_lst_t3'})
)

fig, ax = plt.subplots(figsize=(9, 4))

# Shade rainy season band
for m in RAINY_MONTHS:
    ax.axvspan(m - 0.5, m + 0.5, alpha=0.15, color='royalblue', zorder=0)

ax.plot(
    t3_by_conception['conception_month'],
    t3_by_conception['mean_lst_t3'],
    marker='o', ms=7, color='firebrick', lw=2.0, label='Mean T3 LST'
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
ax.set_xlabel('Conception month', fontsize=11)
ax.set_ylabel('Mean T3 land surface temp. (°C)', fontsize=11)
ax.set_title('Third-trimester heat by conception month — Zimbabwe DHS 2015', fontsize=12)

rainy_patch = mpatches.Patch(color='royalblue', alpha=0.3, label='Rainy season (Nov–Mar)')
ax.legend(handles=[ax.get_lines()[0], rainy_patch])
plt.tight_layout()
plt.show()

peak_c = int(t3_by_conception.loc[t3_by_conception['mean_lst_t3'].idxmax(), 'conception_month'])
print(f"Conception month with highest mean T3 LST: month {peak_c}")
print("Interpretation: babies conceived in Nov–Dec reach T3 during Aug–Oct (hot-dry season).")

---
## Section 3 — Exercise 1 (3 pts): T3 Temperature by Birth Outcome

Among births with a recorded birth weight (`low_birth_weight` is not null), compute the mean third-trimester LST for:
- LBW births (`low_birth_weight == 1`)
- Non-LBW births (`low_birth_weight == 0`)

**Store:**
- `lst_t3_lbw`             — float, 2 decimal places
- `lst_t3_normal`          — float, 2 decimal places
- `temp_diff_lbw_normal`   — float, 2 decimal places  (`lst_t3_lbw - lst_t3_normal`)

In [ ]:
### BEGIN SOLUTION

births_bw = births.dropna(subset=['low_birth_weight']).copy()

lst_t3_lbw    = round(float(births_bw.loc[births_bw['low_birth_weight'] == 1, 'lst_t3'].mean()), 2)
lst_t3_normal = round(float(births_bw.loc[births_bw['low_birth_weight'] == 0, 'lst_t3'].mean()), 2)
temp_diff_lbw_normal = round(lst_t3_lbw - lst_t3_normal, 2)

print(f"Mean T3 LST for LBW births:     {lst_t3_lbw:.2f} °C")
print(f"Mean T3 LST for non-LBW births: {lst_t3_normal:.2f} °C")
print(f"Difference (LBW − normal):      {temp_diff_lbw_normal:+.2f} °C")

if temp_diff_lbw_normal > 0:
    print("\nLBW births experienced slightly higher T3 heat on average.")
else:
    print("\nLBW births experienced slightly lower T3 heat on average — other trimesters may be more important.")

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(lst_t3_lbw, float),          "lst_t3_lbw must be a float"
assert isinstance(lst_t3_normal, float),        "lst_t3_normal must be a float"
assert isinstance(temp_diff_lbw_normal, float), "temp_diff_lbw_normal must be a float"
assert 20.0 <= lst_t3_lbw    <= 35.0,          "lst_t3_lbw out of plausible range"
assert 20.0 <= lst_t3_normal <= 35.0,          "lst_t3_normal out of plausible range"
assert abs(temp_diff_lbw_normal - (lst_t3_lbw - lst_t3_normal)) < 1e-3, \
    "temp_diff_lbw_normal must equal lst_t3_lbw - lst_t3_normal"
### END HIDDEN TESTS

---
## Section 4 — Worked Example: Seasonality of Adverse Outcomes

We plot the monthly prevalence of LBW and neonatal death against birth month and overlay months with the highest mean T3 heat, to explore whether adverse outcomes cluster in particular seasons.

In [ ]:
births_bw2 = births.dropna(subset=['low_birth_weight'])

monthly = (
    births_bw2
    .groupby('birth_month')
    .agg(
        n_births      = ('birth_weight_g', 'count'),
        lbw_rate      = ('low_birth_weight', 'mean'),
        neonatal_mr   = ('neonatal_death', 'mean'),
        mean_lst_t3   = ('lst_t3', 'mean')
    )
    .reset_index()
)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax = axes[0]
ax.bar(monthly['birth_month'], monthly['lbw_rate'] * 100, color='steelblue',
       alpha=0.8, label='LBW rate (%)')
ax2 = ax.twinx()
ax2.plot(monthly['birth_month'], monthly['mean_lst_t3'], color='firebrick',
         marker='s', ms=5, lw=1.5, label='Mean T3 LST (°C)')
ax.set_ylabel('LBW rate (%)', fontsize=10)
ax2.set_ylabel('Mean T3 LST (°C)', fontsize=10, color='firebrick')
ax.set_title('Monthly LBW rate and third-trimester LST', fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax = axes[1]
ax.bar(monthly['birth_month'], monthly['neonatal_mr'] * 1000, color='darkorange',
       alpha=0.8, label='Neonatal MR (per 1,000)')
ax.set_ylabel('Neonatal deaths per 1,000 births', fontsize=10)
ax.set_title('Monthly neonatal mortality rate', fontsize=11)
ax.legend(fontsize=9)

for a in axes:
    a.set_xticks(range(1, 13))
    a.set_xticklabels(month_labels)

axes[1].set_xlabel('Birth month', fontsize=11)
plt.tight_layout()
plt.show()

peak_lbw_month = int(monthly.loc[monthly['lbw_rate'].idxmax(), 'birth_month'])
print(f"Birth month with highest LBW rate: {month_labels[peak_lbw_month-1]} (month {peak_lbw_month})")

---
## Section 5 — Exercise 2 (3 pts): Trimester Rainfall by Birth Outcome

Compute the mean T1, T2, and T3 rainfall separately for LBW births and non-LBW births. Store results as a DataFrame `rain_by_trimester_outcome` with:
- Index: `['t1', 't2', 't3']`
- Columns: `['lbw', 'normal']`

Then identify which trimester shows the largest absolute difference in rainfall between LBW and non-LBW births. Store the trimester name as `max_rain_diff_trimester` (string: `'t1'`, `'t2'`, or `'t3'`).

In [ ]:
### BEGIN SOLUTION

births_bw3 = births.dropna(subset=['low_birth_weight']).copy()
lbw_births    = births_bw3[births_bw3['low_birth_weight'] == 1]
normal_births = births_bw3[births_bw3['low_birth_weight'] == 0]

rain_by_trimester_outcome = pd.DataFrame(
    {
        'lbw'   : [lbw_births['rain_t1'].mean(),    lbw_births['rain_t2'].mean(),    lbw_births['rain_t3'].mean()],
        'normal': [normal_births['rain_t1'].mean(), normal_births['rain_t2'].mean(), normal_births['rain_t3'].mean()],
    },
    index=['t1', 't2', 't3']
).round(2)

print("Mean rainfall (mm/mo) per trimester by birth outcome:")
print(rain_by_trimester_outcome)
print()

abs_diff = (rain_by_trimester_outcome['lbw'] - rain_by_trimester_outcome['normal']).abs()
max_rain_diff_trimester = abs_diff.idxmax()

print(f"Trimester with largest rainfall difference (LBW vs normal): {max_rain_diff_trimester.upper()}")
print(f"  LBW mean:    {rain_by_trimester_outcome.loc[max_rain_diff_trimester, 'lbw']:.2f} mm/mo")
print(f"  Normal mean: {rain_by_trimester_outcome.loc[max_rain_diff_trimester, 'normal']:.2f} mm/mo")

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(rain_by_trimester_outcome, pd.DataFrame), \
    "rain_by_trimester_outcome must be a DataFrame"
assert list(rain_by_trimester_outcome.index) == ['t1', 't2', 't3'], \
    "Index must be ['t1', 't2', 't3']"
assert 'lbw'    in rain_by_trimester_outcome.columns, "Column 'lbw' missing"
assert 'normal' in rain_by_trimester_outcome.columns, "Column 'normal' missing"
assert max_rain_diff_trimester in ('t1', 't2', 't3'), \
    "max_rain_diff_trimester must be 't1', 't2', or 't3'"
assert rain_by_trimester_outcome['lbw'].notna().all(),    "LBW rain values must not be NaN"
assert rain_by_trimester_outcome['normal'].notna().all(), "Normal rain values must not be NaN"
### END HIDDEN TESTS

---
## Section 6 — Worked Example: Drought Exposure Analysis

The `drought_exposed` flag marks births where T2 or T3 rainfall was below 60 mm/month — a proxy for gestational drought stress. We compare LBW rates, birth weights, and neonatal mortality between drought-exposed and non-drought-exposed births.

In [ ]:
# drought_exposed is stored as bool/string — normalise to bool
births['drought_exposed'] = births['drought_exposed'].astype(str).str.lower().isin(['true', '1', 'yes'])

drought_yes = births[births['drought_exposed']]
drought_no  = births[~births['drought_exposed']]

summary = pd.DataFrame({
    'Group'              : ['Drought-exposed', 'Not drought-exposed'],
    'N births'           : [len(drought_yes), len(drought_no)],
    'LBW rate'           : [
        drought_yes['low_birth_weight'].mean(),
        drought_no['low_birth_weight'].mean()
    ],
    'Mean birth wt (g)'  : [
        drought_yes['birth_weight_g'].mean(),
        drought_no['birth_weight_g'].mean()
    ],
    'Neonatal MR'        : [
        drought_yes['neonatal_death'].mean(),
        drought_no['neonatal_death'].mean()
    ],
})

summary['LBW rate']         = summary['LBW rate'].round(3)
summary['Mean birth wt (g)'] = summary['Mean birth wt (g)'].round(1)
summary['Neonatal MR']      = summary['Neonatal MR'].round(3)

print("Drought exposure and birth outcomes — Zimbabwe DHS 2015:")
print(summary.to_string(index=False))
print()
print("Note: drought_exposed = T2 or T3 rainfall < 60 mm/month.")
print("This is an observational comparison; season, wealth, and geography confound the association.")

---
## Section 7 — Exercise 3 (2 pts): Third-Trimester NDVI and Birth Weight

Compute the Pearson correlation between T3 NDVI (`ndvi_t3`) and birth weight in grams (`birth_weight_g`) for births with a recorded birth weight. Store as `r_ndvi_t3_bw` (float, 3 decimal places).

NDVI is a proxy for vegetation cover and food availability — a greener environment during late pregnancy may signal better nutrition access and lower heat stress.

In [ ]:
### BEGIN SOLUTION

births_complete = births.dropna(subset=['birth_weight_g', 'ndvi_t3'])

r_ndvi_t3_bw = round(
    float(births_complete['ndvi_t3'].corr(births_complete['birth_weight_g'])),
    3
)
print(f"Pearson r (T3 NDVI vs birth weight): {r_ndvi_t3_bw:.3f}")
print(f"N = {len(births_complete):,} births with both values recorded")

slope, intercept, *_ = stats.linregress(
    births_complete['ndvi_t3'], births_complete['birth_weight_g']
)
x_line = np.linspace(births_complete['ndvi_t3'].min(), births_complete['ndvi_t3'].max(), 100)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    births_complete['ndvi_t3'], births_complete['birth_weight_g'],
    s=6, alpha=0.25, color='forestgreen', label='Birth'
)
ax.plot(
    x_line, slope * x_line + intercept,
    color='firebrick', lw=2.0, label=f'OLS fit  (r = {r_ndvi_t3_bw})'
)
ax.set_xlabel('Third-trimester NDVI', fontsize=11)
ax.set_ylabel('Birth weight (g)', fontsize=11)
ax.set_title('T3 vegetation cover (NDVI) vs birth weight\nZimbabwe DHS 2015', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(r_ndvi_t3_bw, float),   "r_ndvi_t3_bw must be a float"
assert -1.0 <= r_ndvi_t3_bw <= 1.0,       "correlation must lie in [-1, 1]"
### END HIDDEN TESTS

---
## Recap

| Concept | Key point |
|---|---|
| Exposure timing | T1 = organogenesis; T3 = rapid fetal growth — different windows, different vulnerability |
| Trimester windows | DHS birth month − 9 = conception month; each trimester ≈ 3 calendar months |
| Seasonal patterning | Conception month determines which climate season each trimester falls in |
| Drought flag | T2/T3 rainfall < 60 mm/mo — tracks food insecurity and water stress |
| T3 NDVI and birth weight | Positive correlation expected: greener = more food, lower heat stress |

**Next:** Notebook 05 covers satellite data products, GEE extraction workflows, and temporal aggregation of the pre-extracted monthly climate data.